# Geographic transferability assessment


In [ ]:
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    raise FileNotFoundError("geoctgan")
os.chdir(PACKAGE_ROOT)
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from geoctgan.config import Config

def input_path(label, required=True, root=None):
    raw = input(label + ": ").strip().strip('"').strip("'")
    if not raw:
        if not required:
            return None
        raise ValueError(label)
    path = Path(raw).expanduser()
    if root is not None and not path.is_absolute():
        path = Path(root) / path
    path = path.resolve()
    if required and not path.exists():
        raise FileNotFoundError(label)
    return path

def stored_path(path):
    return str(Path(path).resolve())

@contextmanager
def configuration_file(values):
    with TemporaryDirectory(prefix="geoctgan_") as directory:
        path = Path(directory) / "configuration.json"
        path.write_text(json.dumps(values, indent=2) + "\n", encoding="utf-8")
        yield path


## Ottawa study data


In [ ]:
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd

OTTAWA_INPUT = input_path("Ottawa input ZIP or directory")
DA_ARCHIVE = input_path("Statistics Canada 2021 DA boundary ZIP")
DA_POPULATION = input_path("Ottawa 2021 DA population CSV")
OUTPUT_PARENT = input_path("Output directory", required=False) or (PACKAGE_ROOT / "Output")
RUN_ROOT = OUTPUT_PARENT / "Ottawa"
PREPARED_DIR = RUN_ROOT / "PreparedInputs"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
START_YEAR, END_YEAR = 2018, 2024
METRIC_CRS = "EPSG:32618"
MODEL_SEEDS = [42, 52, 62]
PUBLIC_COLUMNS = ["SYNTHETIC_ID", "OCC_YEAR", "OCC_MONTH", "OCC_DAY", "OCC_DOW", "OCC_HOUR", "MCI_CATEGO", "OFFENCE", "DIVISION", "LONG_WGS84", "LAT_WGS84"]

if OTTAWA_INPUT.is_file():
    if OTTAWA_INPUT.suffix.casefold() != ".zip":
        raise ValueError("Ottawa input ZIP or directory")
    source_root = (RUN_ROOT / "RawInput").resolve()
    source_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(OTTAWA_INPUT) as archive:
        for member in archive.infolist():
            target = (source_root / member.filename).resolve()
            if target != source_root and source_root not in target.parents:
                raise ValueError("Archive member path")
        archive.extractall(source_root)
else:
    source_root = OTTAWA_INPUT

CRIME_PATH = input_path("Ottawa offence shapefile", root=source_root)
ROAD_PATH = input_path("Ottawa road-centreline shapefile", root=source_root)
EXTENT_PATH = input_path("Ottawa neighbourhood-extent shapefile", root=source_root)
for path, label in ((CRIME_PATH, "Offence shapefile"), (ROAD_PATH, "Road shapefile"), (EXTENT_PATH, "Extent shapefile")):
    if not path.is_file() or path.suffix.casefold() != ".shp":
        raise ValueError(label)


## Event and road-network preparation


In [ ]:
crime = gpd.read_file(CRIME_PATH)
required_crime = {"OCC_DATE", "OCC_HOUR", "OFF_SUM", "OFF_CATEG", "DIVISION"}
missing_crime = sorted(required_crime.difference(crime.columns))
if missing_crime:
    raise KeyError(tuple(missing_crime))
occurrence_date = pd.to_datetime(crime["OCC_DATE"], errors="coerce")
raw_hour = pd.to_numeric(crime["OCC_HOUR"], errors="coerce")
if raw_hour.isna().any() or (raw_hour % 100).ne(0).any():
    raise ValueError("OCC_HOUR")
hour = (raw_hour / 100).astype(int)
if not hour.between(0, 23).all():
    raise ValueError("OCC_HOUR")
events = gpd.GeoDataFrame(
    {
        "EVENT_PK": [f"OTTAWA_{position:09d}" for position in range(len(crime))],
        "OCC_DATE": occurrence_date,
        "OCC_HOUR": hour,
        "MCI_CATEGO": crime["OFF_SUM"].astype("string").fillna("<MISSING>"),
        "OFFENCE": crime["OFF_CATEG"].astype("string").fillna("<MISSING>"),
        "DIVISION": crime["DIVISION"].astype("string").fillna("<MISSING>"),
        "SOURCE_ROW": np.arange(len(crime), dtype=np.int64),
    }, geometry=crime.geometry, crs=crime.crs,
)
valid = (
    events.geometry.notna() & ~events.geometry.is_empty
    & events.geometry.geom_type.eq("Point") & events["OCC_DATE"].notna()
    & events["OCC_DATE"].dt.year.between(START_YEAR, END_YEAR)
)
events = events.loc[valid].copy().to_crs(4326)
if events.empty or not events["EVENT_PK"].is_unique:
    raise ValueError("EVENT_PK")

roads_raw = gpd.read_file(ROAD_PATH)
required_road = {"RD_SEGMENT", "CURRENT_ST", "SUBTYPE_TE", "SUBCLASS", "OWNERSHIP", "FULL_ROADN", "geometry"}
missing_road = sorted(required_road.difference(roads_raw.columns))
if missing_road:
    raise KeyError(tuple(missing_road))
roads_raw = roads_raw.loc[
    roads_raw["CURRENT_ST"].astype(str).isin(["Operational", "Open to Traffic"])
    & roads_raw.geometry.notna() & ~roads_raw.geometry.is_empty
    & roads_raw.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy().to_crs(METRIC_CRS)
if not roads_raw.geometry.geom_type.eq("LineString").all():
    raise ValueError("LineString")
roads = gpd.GeoDataFrame(
    {
        "source_road_id": roads_raw["RD_SEGMENT"].astype("string"),
        "road_subtype": roads_raw["SUBTYPE_TE"].astype("string"),
        "road_class": roads_raw["SUBCLASS"].astype("string"),
        "ownership": roads_raw["OWNERSHIP"].astype("string"),
        "current_status": roads_raw["CURRENT_ST"].astype("string"),
        "road_name": roads_raw["FULL_ROADN"].astype("string"),
    }, geometry=roads_raw.geometry, crs=roads_raw.crs,
).reset_index(drop=True)
if roads.empty or roads["source_road_id"].isna().any() or roads["source_road_id"].duplicated().any():
    raise ValueError("RD_SEGMENT")

extent = gpd.read_file(EXTENT_PATH)
extent = extent.loc[extent.geometry.notna() & ~extent.geometry.is_empty].copy().to_crs(METRIC_CRS)
if (~extent.geometry.is_valid).any():
    extent.geometry = extent.geometry.make_valid()
SPATIAL_EXTENT_M = tuple(float(value) for value in extent.total_bounds)

EVENTS_ZIP = PREPARED_DIR / "ottawa_events_prepared.zip"
NETWORK_ZIP = PREPARED_DIR / "ottawa_network_prepared.zip"
for frame, filename, layer, destination in (
    (events, "ottawa_events.gpkg", "events", EVENTS_ZIP),
    (roads, "ottawa_municipal_roads.gpkg", "roads_edges", NETWORK_ZIP),
):
    target = PREPARED_DIR / filename
    frame.to_file(target, layer=layer, driver="GPKG", mode="w")
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(target, arcname=filename)


## Official 2021 dissemination areas


In [ ]:
population = pd.read_csv(DA_POPULATION, dtype={"DAUID": "string"})
required_population = {"DAUID", "Population, 2021", "City", "Source"}
missing_population = sorted(required_population.difference(population.columns))
if missing_population:
    raise KeyError(tuple(missing_population))
population["DAUID"] = population["DAUID"].str.strip().str.zfill(8)
population["Population, 2021"] = pd.to_numeric(population["Population, 2021"], errors="raise").astype("int64")
if (
    len(population) != 1392 or population["DAUID"].duplicated().any()
    or not population["City"].astype(str).eq("Ottawa").all()
    or int(population["Population, 2021"].sum()) != 1017449
):
    raise ValueError("Ottawa DA population")
with zipfile.ZipFile(DA_ARCHIVE) as archive:
    shape_members = sorted(name for name in archive.namelist() if name.casefold().endswith(".shp"))
if len(shape_members) != 1:
    raise ValueError("DA boundary archive")
da_candidates = gpd.read_file(f"zip://{DA_ARCHIVE}!{shape_members[0]}", where="DAUID LIKE '3506%'", engine="pyogrio")
da_candidates["DAUID"] = da_candidates["DAUID"].astype("string").str.strip().str.zfill(8)
ottawa_da = da_candidates.merge(population, on="DAUID", how="inner", validate="one_to_one")
ottawa_da = gpd.GeoDataFrame(ottawa_da, geometry="geometry", crs=da_candidates.crs)
if len(ottawa_da) != 1392 or set(ottawa_da["DAUID"]) != set(population["DAUID"]):
    raise ValueError("Ottawa DAUID")
invalid = ~ottawa_da.geometry.is_valid
if invalid.any():
    ottawa_da.loc[invalid, "geometry"] = ottawa_da.loc[invalid, "geometry"].make_valid()
if (
    ottawa_da.geometry.isna().any() or ottawa_da.geometry.is_empty.any()
    or (~ottawa_da.geometry.is_valid).any()
    or not ottawa_da.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all()
):
    raise ValueError("DA geometry")
ottawa_da = ottawa_da.to_crs(METRIC_CRS).sort_values("DAUID").reset_index(drop=True)
DA_BOUNDARIES = PREPARED_DIR / "ottawa_da.gpkg"
ottawa_da.to_file(DA_BOUNDARIES, layer="da_2021", driver="GPKG", mode="w")


## Transfer protocol


In [ ]:
from geoctgan.data import create_outer_split, load_events

setup_cfg = Config(
    BASE_DIR=stored_path(PREPARED_DIR), OUTPUT_DIR=stored_path(RUN_ROOT),
    ZIP_EVENTS=stored_path(EVENTS_ZIP), ZIP_NETWORK=stored_path(NETWORK_ZIP),
    EVENTS_MEMBER="ottawa_events.gpkg", NETWORK_MEMBER="ottawa_municipal_roads.gpkg", NETWORK_LAYER="roads_edges",
    START_YEAR=START_YEAR, END_YEAR=END_YEAR, CRS_GEO="EPSG:4326", CRS_METRIC=METRIC_CRS,
    REQUIRE_AUTHORITATIVE_TOPOLOGY=False, NODE_SNAP_TOL_M=0.75, MAX_ALLOWED_ISOLATED_SEGMENTS=0,
    SNAP_MAX_M=50.0, EMB_DIM=16, N_WALKS=15, WALK_LEN=10, COHORT_KIND="global",
    W2V_WINDOW=10, W2V_EPOCHS=6, W2V_WORKERS=1, EPOCHS=300,
    SENSITIVITY_SCREEN_SEED=42, SENSITIVITY_SEEDS=MODEL_SEEDS, DEFINITIVE_SEEDS=MODEL_SEEDS,
    PUBLIC_COLUMNS=PUBLIC_COLUMNS, REQUIRE_SELECTED_CONFIG=False,
    REQUIRE_REUSED_OUTER_SPLIT=True, PUBLIC_RELEASE_ENABLED=False,
)
setup_cfg.validate()
reference_dir = RUN_ROOT / "Configuration" / "Data" / "Reference"
reference_dir.mkdir(parents=True, exist_ok=True)
clean_events = load_events(setup_cfg, str(reference_dir))
outer_train, outer_test, split_manifest = create_outer_split(setup_cfg, clean_events, str(reference_dir))
setup_cfg.OUTER_SPLIT_MANIFEST_PATH = stored_path(split_manifest)


## Multi-seed generation


In [ ]:
from geoctgan.pipeline import run_pipeline

with configuration_file(setup_cfg.as_dict()) as SELECTED_CONFIG:
    definitive_values = {
        "SELECTED_CONFIG_PATH": stored_path(SELECTED_CONFIG), "REQUIRE_SELECTED_CONFIG": True,
        "OUTPUT_DIR": stored_path(RUN_ROOT), "EPOCHS": 300, "ENABLE_GPU": True,
        "DEFINITIVE_SEEDS": MODEL_SEEDS, "PUBLIC_RELEASE_ENABLED": True,
        "PUBLIC_RELEASE_CAPACITY_MODE": "upper_quota", "ALLOW_EXACT_QUOTA_PUBLIC_RELEASE": False,
        "RESUME_TRAINING": True, "RESUME_GENERATION": True, "RESUME_CANN": True,
    }
    with configuration_file(definitive_values) as DEFINITIVE_CONFIG:
        generation = run_pipeline(str(DEFINITIVE_CONFIG), seeds=MODEL_SEEDS, run_dir=RUN_ROOT / "Generation")
GENERATION_SUMMARY = RUN_ROOT / "Generation" / "Metadata" / "run_summary.json"


## Held-out fidelity and privacy diagnostics


In [ ]:
from geoctgan.evaluation import EvaluationConfig, run_evaluation

evaluation = run_evaluation(
    GENERATION_SUMMARY,
    output_parent=OUTPUT_PARENT,
    evaluation_dir=RUN_ROOT / "Evaluation",
    da_boundaries=DA_BOUNDARIES,
    marginalization_csv=None,
    benchmark_manifest=None,
    config=EvaluationConfig(
        metric_crs=METRIC_CRS,
        spatial_extent_m=SPATIAL_EXTENT_M,
        heavy_privacy_datasets=[
            f"GeoCTGAN-{profile} | seed {seed}"
            for profile in ("Full", "Priv", "UpperQuota")
            for seed in MODEL_SEEDS
        ],
        heavy_privacy_columns=[column for column in PUBLIC_COLUMNS if column != "SYNTHETIC_ID"],
        privacy_native_subsample_rows=10000,
        privacy_native_iterations=10,
        privacy_profile="ottawa",
        strict_completion=True,
    ),
)
